## Preprocessing

For the conduction of the original study, a Biosemi Active-Two system with 64 channels and a sampling rate of 2048 Hz was used with 10-20 electrode placement. The authors of the study visually inspected the EEG data of each participant in order to find noisy channels with artifacts, on whom interpolation was applied in a later step. According to the [original preprocessing MATLAB file](https://osf.io/yjxkn/files/5dq83), only for the visual inspection filters were applied.

After identifying the bad channels, the authors epoched the data with intervals of 200ms before the decision screen was presented to the participants to 5000 ms after the onset of the decision screen. The noisy channels where then interpolated by using functionality of the open-source MATLAB toolbox FieldTrip. After the interpolation, the data was downsampled to 256 Hz, arguably to reduce the memory demands. 

Additionally, in the original MATLAB file of the decoding step, the epoched, interpolated and downsampled data was re-referenced to the common average. Notably, in the paper, the authors claimed to have started the preprocessing with re-referencing the data, which was not the case according to the MATLAB preprocessing script. Re-referencing with the common average as the very first step would also mean that the noise from the "bad" channels would contribute to it, which would mean that noise would be added to all of the channels.

## Reproduction of the preprocessing

The first step was to import the dataset, which we did by using functionality of the MNE-Python package. One inital problem we had was that the electrodes in the dataset had Biosemi-specific names, which we first could not map to the 10-20 system. The authors of the study only added the `biosemi64.mat` file to their repository, which contains the 3D coordinates of the leads, but not the `biosemi64.lay` containing the mapping to the 10-20 electrode names, which we could find in the GitHub repository of the FieldTrip toolbox. As a quick sanity check, we plotted the 3D electrode positions as well as the 10-20 mapping together with their row indices. The first image shows all the biosemi64 electrodes with their row indices. Together with the second image we could verify that the rows in the two files align with each other.

In [ ]:
import pathlib
import src.layout_check


src.layout_check.plot_3D_layout(pathlib.Path("data/biosemi64.mat"))
src.layout_check.plot_biosemi_layout_to_10_20(pathlib.Path("src/biosemi64.lay"))

### Dataset import

According to the paper, three subjects were excluded from the analysis. For the reproduction, we therefore left them out too. From the original MATLAB preprocessing script, we could learn that subjects 10, 23 and 24 were excluded.

The `get_from` method of our "main" class `BidsDataset` (at least for the preprocessing) constructs a `BidsDataset` instance by reading participant metadata from the `participants.tsv` file located in the dataset's root directory. It iterates over all entries, excludes the predefined subjects, and instantiates a `Subject` object for each remaining pair of participants. The resulting list of subjects, together with the input and output paths, is then used to initialize and return the `BidsDataset` instance. The `Subject` class captures two players and the subject's events and has the functionality to preprocess the EEG data of one individual pair of players of the dataset.

In [ ]:
import pathlib
import src.dataset
import gc

dataset_path = pathlib.Path("ds006761")
output_dir = pathlib.Path("data/results/preprocesing")

dataset = src.dataset.BidsDataset.get_from(dataset_path, output_dir, exclude_subjects=["sub-10", "sub-23", "sub-24"])

### Read the Subjects raw EG data and separate the two players' raw EEG data

Next, the `read_raw_eeg_data` method loads a subject’s raw EEG recording. Internally, it uses `mne_bids.read_raw_bids` for this. The appropriate `mne_bids.BIDSPath` is constructed based on the subject ID and task. The resulting raw object contains both players EEG data and therefore needs to be separated further, which is done by the `prepare_players` method. The `prepare_players` method separates the joint raw into two individual player raw EEG objects by selecting channel subsets corresponding to each player’s prefix. According to the MATLAB preprocessing file, there was a mixup regarding the player prefixes, which is why player 1 has prefix "2-". For each player, the method verifies the channel completeness, extracts the relevant channels, renames them to standard 10–20 system labels, assigns EEG channel types and applies a BioSemi montage for spatial electrode configuration. The method returns two MNE `Raw` objects, one for each player. Afterwards, the joint raw is immediately deleted, as it is not needed anymore consumes a considerable amount of memory.

In [ ]:
sub1 = dataset.subjects[0]

raw = sub1.read_raw_eeg_data(dataset.bids_root)
raw_p1, raw_p2 = sub1.prepare_players(raw)

del raw
gc.collect()

### Creation of epochs for each player

The `epoch_players` method method creates epochs from each player's raw data and uses MNE functionality internally for it. The epochs are defined based on stimulus onset sample indices, corresponding to the start of each decision phase in the rock–paper–scissors task. Consistent with the original MATLAB preprocessing pipeline, the epoch boundaries are defined via the discrete sample indices rather than directly in seconds. Each epoch spans from 0.2 seconds before stimulus onset to 5.0 seconds after stimulus onset. This allows capturing both pre-stimulus activity and the neural responses that appear as a reaction to the stimulus.

In our reproduction the explicit conversion between time in seconds and sample indices is notable. The desired epoch window is first converted into integer sample counts using the ceiling operation. The window boundaries are then converted back into seconds to comply with MNE, which only takes seconds for the epoching. With this approach we tried to ensure that the epoch boundaries closely replicate the trial definition used in the original MATLAB implementation. As a result, the same discrete data samples are selected for each epoch. As a baseline correction is applied in a later step right before the decoding and not as part of the preprocessing part (it is also like this in the paper's processing pipeline), internally, the `mne.Epochs` is called with `baseline=None`.

In [ ]:
epochs_p1 = sub1.epoch_players(raw_p1)
epochs_p2 = sub1.epoch_players(raw_p2)

del raw_p1, raw_p2
gc.collect()

### Interpolation of the bad channels

In the original preprocessing pipeline, the interpolation of the channels identified as bad comes next. In our reproduction, this is achieved with the `interpolate` method, which performs a reconstruction of the bad channels using a custom implementation that closely replicates FieldTrip's `ft_channelrepair(method='weighted')` that is used in the MATLAB preprocessing script, as no equivalent function directly exists in MNE.

Specifically, the method identifies channels marked as bad in the respective players metadata and reconstructs their signals as weighted averages of neighboring good channels. Neighbors are defined based on spatial proximity using the BioSemi 3D electrode layout from the `biosemi64.mat` file. The weights are computed as the inverse of the Euclidean distance between electrodes to ensure that closer electrodes contribute more strongly. For each bad channel, its original signal is replaced by this weighted combination, while good channels remain unchanged.

To apply the transformation in an efficient way, the method uses `numpy` functionality to construct a full channel-by-channel repair matrix, where each row encodes how a given channel should be reconstructed. This matrix is then applied to all epochs simultaneously via matrix multiplication. If a bad channel has no valid neighboring electrodes, its data is set to `NaN`, which again matches the FieldTrip behavior (we took a look at the original implementation).

To inspect the effect of the interpolation, a bad channel and its neighbors are plotted before and after the interpolation. In the case of player 2 of subject 1, T8 was identified as a bad channel. According to the 10-20 system, among the neighboring electrodes (electrodes with close proximity) would be FT8, TP8 and C6. In the plot below, we can clearly see that compared to the other channels, the signal of T8 has a much higher variance and with more high-frequency jitter and spikes. Note, that it is a know issue in MNE that plots get printed twice in Jupyter Notebooks.

In [ ]:
epochs_p2.plot(picks=["FT8", "C6", "TP8", "T8"], scalings={"eeg": 100e-6}, n_epochs=1)

After running the interpolation as decribed above, we can see that $T8$ appears to look a lot more like the neighboring electrodes with less jitter and spikes.

In [ ]:
sub1.interpolate(epochs_p1, sub1.player1, "Player 1")
sub1.interpolate(epochs_p2, sub1.player2, "Player 2")
epochs_p2.plot(picks=["FT8", "C6", "TP8", "T8"], scalings={"eeg": 100e-6}, n_epochs=1)

### Resampling to 256 Hz

The resampling step reduces the temporal resolution of the epoched EEG data from the original sampling rate of 2048 Hz to a lower rate of 256 Hz. From a signal-processing perspective, the EEG signals of interest are typically lower than 50 Hz (8-13 Hz alpha waves, 13-30Hz beta waves, > 30Hz Gamma waves). According to the Shannon-Nyquist theorem, a sampling rate of 256 Hz is sufficient to represent frequencies up to 128 Hz, which therefore covers the relevant bandwidth. Therefore, downsampling should not lead to a meaningful loss of information, while at the same time reduces the memory demands. According to the MNE-Bids documentation, the `resample` method additionally applies a lowpass filter at the Nyquist frequency (which is at 256 / 2 = 128 Hz in this case) to avoid aliasing effects. For a sanity check, the ERP of the F4 channel of player 1 of the first subject is visualized before and after the application of the resampling. Only one channel is visualized, because the data is not yet rereferenced and baselined.

In [ ]:
evoked = epochs_p1.average()
evoked.plot(picks=["F4"], scalings={"eeg": 100-6})

### Resampling to 256 Hz

The resampling step reduces the temporal resolution of the epoched EEG data from the original sampling rate of 2048 Hz to a lower rate of 256 Hz. From a signal-processing perspective, the EEG signals of interest are typically lower than 50 Hz (8-13 Hz alpha waves, 13-30Hz beta waves, > 30Hz Gamma waves). According to the Shannon-Nyquist theorem, a sampling rate of 256 Hz is sufficient to represent frequencies up to 128 Hz, which therefore covers the relevant bandwidth. Therefore, downsampling should not lead to a meaningful loss of information, while at the same time reduces the memory demands. According to the MNE-Bids documentation, the `resample` method additionally applies a lowpass filter at the Nyquist frequency (which is at 256 / 2 = 128 Hz in this case) to avoid aliasing effects. For a sanity check, the ERP of the F4 channel of player 1 of the first subject is visualized before and after the application of the resampling. Only one channel is visualized, because the data is not yet rereferenced and baselined.

In [ ]:
if not epochs_p1.preload:
    epochs_p1.load_data()
epochs_p1.resample(256, verbose=False)

if not epochs_p2.preload:
    epochs_p2.load_data()
epochs_p2.resample(256, verbose=False)


In [ ]:
evoked = epochs_p1.average()
evoked.plot(picks=["Cz"], scalings={"eeg": 100-6})

As it can be seen above, the ERP of the F4 channel looks a bit smoother than before, as it would be expected after the anti-aliasing filtering that was applied as part of the downsampling. As the data is not yet rereferenced and baselined, there appears a drift and the units are still off. Still, the task related structure is visible in the signal, with peaks in the decision, response and feedback phases. This indicates that the data is not corrupted by the applied preprocessing steps.

### Save the Preprocessed players' epochs

Lastly, the preprocessed EEG data is saved to disk for each player, so that it can be used in the subsequent decoding step.

In [ ]:
sub1.save(epochs_p1, epochs_p2, output_dir)

del epochs_p1, epochs_p2
gc.collect()

## Run the preprocessing for every subject

The preprocessing pipeline can be executed for each subject by calling the `preprocess` method of the `BidsDataset` class. Note that it is not recommended to run the preprocessing inside this Jupyter Notebook, as it takes quite some time to finish in general and using the Jupyter Kernel further slows down the process. Please use the provided `main.py` to run the preprocessing and other steps of the pipeline. 

In [ ]:
# would run the preprocessing for every subject of the dataset
dataset.preprocess()

### Contribution to preprocessing

Preprocessing is often assumed to improve decoding performance by enhancing the signal quality. However, as highlighted in the paper ["How EEG preprocessing shapes decoding performance"](https://www.nature.com/articles/s42003-025-08464-3), it is not necessarily a silver bullet in decoding analysis. In particular, the paper demonstrates that certain preprocessing steps can remove not only noise but also information that is predictive for the classifier, since artifacts themselves may systematically covary with the experimental conditions. For example, in the context of the rock–paper–scissors task, it would be plausible that muscle activity or eye blinks occur more frequently after a player has lost, which would introduce artifacts that a classifier could exploit. Consequently, aggressive artifact removal may reduce decoding performance by eliminating these non-neural signals. The paper therefore proposes a cautious approach for preprocessing and suggests that relatively simple steps such as filtering and detrending can be beneficial, while more complex artifact correction procedures should be evaluated carefully in terms of their impact on decoding. By taking these considerations into account, we tried to explore different preprocessing strategies: band-pass filtering using a relatively high high-pass and low low-pass cutoff, as well as a combination of filtering and detrending, which produced the best results in the paper, and evaluation of artifact handling by comparing interpolation of channels marked as bad with the alternative of simply dropping those channels. The effects of the described additional preprocessing steps will again be shown for one player in this Notebook. To run the preprocessing for all of the subjects, please use the functionality provided in the `main.py`

#### Bandpass filter

We first applied a band-pass filter to the continuous, raw EEG data of each player using a high-pass cutoff of 1 Hz and a low-pass cutoff of 35 Hz. This frequency range was chosen because this interval contains the frequencies of interest for cognitive EEG, including alpha activity (approximately 8–13 Hz), as well as theta (4–7 Hz) and beta (13–30 Hz) bands, while  the effects of very slow drifts and high-frequency noise such as muscle activity are weakened. We performed the filtering on the continuous raw data rather than on the epoched data. This choice is important because filtering assumes temporal continuity. Applying it to already segmented epochs introduces artificial boundaries that can lead to edge artifacts and distortions near the beginning and end of each trial. By filtering the raw signal before epoching the data, these boundary effects are minimized, which should result in a more stable signal representation.

For the filtering, we used the MNE function `filter`, which by default uses a FIR filter.

In [1]:
# Get sub1's raw data again
sub1 = dataset.subjects[0]

raw = sub1.read_raw_eeg_data(dataset.bids_root)
raw_p1, raw_p2 = sub1.prepare_players(raw)

del raw
gc.collect()

NameError: name 'dataset' is not defined

In [ ]:
# apply the filtering
raw_p1_filtered = raw_p1.copy().filter(l_freq=1.0, h_freq=35.0)

psd_before = raw_p1.compute_psd(fmax=128)
psd_after = raw_p1_filtered.compute_psd(fmax=128)

print("before:")
psd_before.plot(average=True, show=False)


In [ ]:
print("after:")
psd_after.plot(average=True)

The power spectral density graph of the unfiltered raw of player 1 shows a spike at 50 Hz and a spike at 100 Hz, which is a harmonic of 50 Hz. According to the MNE tutorial on artifact detection, this is characteristic for powerline interference. After applying the band-pass filter, those peaks are considerably weakened, as the frequencies above the upper cutoff are suppressed.

##### Decoding results of preprocessing with filtering

A comparison of decoding performance with (left figure) and without (default pipeline from the paper, right figure) band-pass  filtering (1–35 Hz) reveals that filtering partially led to a slight increase in the decoding accuracy, especially during the feedback phase of the opponent's response plot, while the overall temporal structure of the results remained unchanged. However, improvements were not consistent across all conditions. In some cases, the decoding performance was comparable or even slightly reduced after filtering, for example during the response phase of the opponent’s previous response condition (plot (d)). The peak decoding times and condition-specific patterns are consistent across both preprocessing approaches, which shows that the observed effects are not introduced by filtering artifacts. These findings support the notion that while band-pass filtering can enhance decoding performance, it does not fundamentally alter the underlying decodable information, which appears to be robustly present in the data even without such preprocessing.

<img src="images/group_lda_cosmo_bandpass_1_35.png" width="400"> <img src="images/LDA.png" width="400">